# 🛡️ Support Integrity Auditor (SIA) — End-to-End Walkthrough

Three-stage pipeline on the real **Customer Support Tickets CRM** dataset (20,000 tickets):

1. **Self-supervised pseudo-labels** — fuse severity signals, calibrate to the priority scale, derive mismatch.
2. **Supervised classifier** — fine-tuned DeBERTa-v3-small (or a no-GPU TF-IDF+MLP fallback).
3. **Evidence Dossier** — deterministic, hallucination-verified explanations.

No ground-truth mismatch labels are used anywhere.


In [ ]:
import json, pandas as pd
from src import config as C, data as D, signals as S, model as M, dossier as DOSS, metrics as MET
pd.set_option('display.max_colwidth', 70)


## Stage 0 — Data
Put the Kaggle CSV at `data/tickets.csv`. The loader maps the real column names
(`Priority_Level`, `Issue_Category`, `Resolution_Time_Hours`, …), strips the `Hi Support,`
greeting, and reduces each ticket to its core issue sentence.


In [ ]:
df = D.load_prepared('data/tickets.csv')
print(df.shape, '| resolution known:', f'{df.resolution_known.mean():.1%}')
print('assigned priority:', df.priority.value_counts().to_dict())
df[['ticket_id','priority','text','resolution_hours']].head()


## Stage 1 — Self-supervised pseudo-labels
Signals → continuous fused score → **quantile calibration** onto the 0..3 scale
(so inferred severity is comparable to the assigned priority) → binary mismatch.
Use `signals=('lex','rt','emb')` to add the semantic signal; `('lex','rt')` is the fast default.


In [ ]:
pl, rt_scaler, embedder, calibrator = S.generate_pseudo_labels(df, signals=('lex','rt'))
print('calibrator cuts:', [round(c,3) for c in calibrator.cuts])
print('inferred dist :', pl.inferred_sev.value_counts().to_dict())
print('mismatch rate :', f'{pl.mismatch.mean():.1%}')
print('types         :', pl[pl.mismatch==1].mismatch_type.value_counts().to_dict())
pl[['ticket_id','priority','inferred_sev','delta','mismatch','mismatch_type']].head()


In [ ]:
# Signal agreement: lex and rt are near-independent -> fusing them adds information
S.signal_agreement(pl, 'sev_lex', 'sev_rt')


## Stage 2 — Supervised classifier
Input = text **+** structured metadata (assigned priority, channel, type, resolution band).
`assigned_priority` is a legitimate input (mismatch is defined relative to it). Mismatch is an
XOR-like interaction, so a **nonlinear** model is required. Imbalance is handled via weighted
loss (DeBERTa) / minority oversampling (MLP).


In [ ]:
from sklearn.model_selection import train_test_split
train, temp = train_test_split(pl, test_size=0.30, stratify=pl.mismatch, random_state=C.SEED)
val, test  = train_test_split(temp, test_size=0.50, stratify=temp.mismatch, random_state=C.SEED)
print(f'train={len(train)} val={len(val)} test={len(test)}')
print('example serialized input:'); print(' ', M.serialize_input(train.iloc[0]))


In [ ]:
from pathlib import Path
outdir = Path(C.ARTIFACTS_DIR)
# backend='deberta' for the spec model (GPU); 'baseline' runs anywhere
backend = M.train_classifier(train, val, outdir, backend='baseline')
print('trained backend:', backend)


### Evaluate against the spec thresholds
accuracy ≥ 0.83 · macro-F1 ≥ 0.82 · per-class recall ≥ 0.78 (both classes).


In [ ]:
probs = M.predict_proba(test, outdir, backend)
preds, conf = M.proba_to_pred(probs)
m = MET.evaluate(test.mismatch, preds)
print(MET.report(test.mismatch, preds))
print({k: round(m[k],4) for k in ['accuracy','macro_f1','recall_consistent','recall_mismatch']})
print('THRESHOLDS:', m['passes'])


## Stage 3 — Evidence Dossier (verified hallucination-free)
Built deterministically from real fields; `verify_dossier()` proves every evidence value is
traceable to a source column.


In [ ]:
test = test.copy(); test['prediction'] = preds; test['confidence'] = conf.round(3)
flagged = test[test.prediction==1]
violations = sum(len(DOSS.verify_dossier(DOSS.build_dossier(r, r.confidence), r)) for _, r in flagged.iterrows())
print(f'flagged={len(flagged)} | total hallucination violations={violations}')
print(json.dumps(DOSS.build_dossier(flagged.iloc[0], flagged.iloc[0].confidence), indent=2))


## Adversarial robustness (bonus)
The semantic signal flags crises with **no** alarming keywords; negation defuses
keyword-stuffed false alarms. (Requires `sentence-transformers`.)


In [ ]:
emb = S.EmbeddingScorer().load()
if emb.available:
    for t in ['the platform is unreachable and customers are furious',
              'this is not urgent at all, just a quick question']:
        print(round(emb.score(t)[0],2), '::', t)
else:
    print('install sentence-transformers to run the embedding signal')


## Next steps
- Spec model on GPU: `python train_pipeline.py --data data/tickets.csv --backend deberta --epochs 4`
- Batch inference + dossiers: `python predict.py --input data/tickets.csv`
- Web app: `streamlit run app.py`
